In [26]:
import fastplotlib as fpl
import os
import masknmf
import sys
import numpy as np

from ipywidgets import HBox, VBox
import math
import torch

##In the existing folder
import iblwfci_vis
import iblwfci_utils

from tqdm import tqdm
from one.api import ONE
from brainbox.io.one import SessionLoader
import matplotlib.pyplot as plt
import math
from tqdm import tqdm
one = ONE()

import matplotlib.pyplot as plt
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
"""
The two datasets from FD that we will process are: 
(1) 71ceb3d4-ca68-4380-8fe7-9f63d26222f6 (This is the existing FD_24/2023-06-07/001 data that I had already downloaded) (outputs are in the 007 folder)
(2) 8df7b200-e44c-4c67-82e9-2666ba05d649 (This is the  FD_24/2023-06-08/001 data) (outputs are in the 008 folder)
"""

'\nThe two datasets from FD that we will process are: \n(1) 71ceb3d4-ca68-4380-8fe7-9f63d26222f6 (This is the existing FD_24/2023-06-07/001 data that I had already downloaded) (outputs are in the 007 folder)\n(2) 8df7b200-e44c-4c67-82e9-2666ba05d649 (This is the  FD_24/2023-06-08/001 data) (outputs are in the 008 folder)\n'

In [31]:
## Load a bunch of (potentially) relevant data
eid = "71ceb3d4-ca68-4380-8fe7-9f63d26222f6"
sl = SessionLoader(eid = eid, one=one)
sl.load_trials()
print(sl.trials.keys())
stim_ontimes = sl.trials['stimOn_times']
firstMovement = sl.trials['firstMovement_times'].to_numpy()
firstMovement = firstMovement[~np.isnan(firstMovement)]
feedback_times = sl.trials['feedback_times'].to_numpy()
sl.load_motion_energy() ## Load Motion Energy seems to fail here, not sure why 

sl.load_wheel()
imaging_times = one.load_dataset(eid, "imaging.times.npy", collection="alf/widefield")

/data/home/app2139/wfield/wfieldvenv/lib/python3.11/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "2026-01-16", ""
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


Index(['goCueTrigger_times', 'included', 'intervals_bpod_0',
       'intervals_bpod_1', 'quiescencePeriod', 'stimOffTrigger_times',
       'stimOff_times', 'stimOnTrigger_times', 'goCue_times', 'response_times',
       'choice', 'stimOn_times', 'contrastLeft', 'contrastRight',
       'feedback_times', 'feedbackType', 'rewardVolume', 'probabilityLeft',
       'firstMovement_times', 'intervals_0', 'intervals_1'],
      dtype='object')


/data/home/app2139/wfield/wfieldvenv/lib/python3.11/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "", "2026-01-20"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)
/data/home/app2139/wfield/wfieldvenv/lib/python3.11/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "", "2026-01-20"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)
/data/home/app2139/wfield/wfieldvenv/lib/python3.11/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "", "2026-01-20"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


In [32]:
## Load the appropriate results from processing this data
parent_path = "/data/lab/IBL_Alyx/churchlandlab_ucla/Subjects/FD_24/2023-06-07/001/raw_widefield_data"
u_path = os.path.join(parent_path, "U.npy")
svt_path = os.path.join(parent_path, "SVT.npy")
svtcorr_path = os.path.join(parent_path, "SVTcorr.npy")
frames_avg_path = os.path.join(parent_path, "frames_average.npy")

manual_mask = np.load(os.path.join(parent_path, "manual_mask.npy"))

In [33]:
## Load Amol pipeline results: 
hemocorr_pmd = masknmf.PMDArray.from_hdf5("felicia_jan_26_007/hemocorr.hdf5")
hemocorr_pmd.to('cuda')
hemocorr_pmd.rescale = True

In [34]:
## Load Joao pipeline results:
_, _, joao_hemocorr = iblwfci_utils.load_joao_results(u_path,
                  svt_path,
                  svtcorr_path,
                  frames_avg_path,
                  functional_channel = 0)

joao_hemocorr.rescale = True
joao_hemocorr.to('cuda')

In [10]:
## Now load the appropriate wheel velocity data

In [35]:
def extract_time_aligned_behavior(imaging_times, functional_channel, behavior_times, behavior_values):
   
    functional_times = imaging_times[functional_channel::2]

    time_subset = np.logical_and(behavior_times < np.amax(functional_times), behavior_times > np.amin(imaging_times))
    behavior_times = behavior_times[time_subset]
    behavior_values = behavior_values[time_subset]
    behavior_indices = np.searchsorted(behavior_times, functional_times)
    behavior_indices[behavior_indices >= behavior_values.shape[0]] = behavior_values.shape[0] - 1
    behavior_indices[behavior_indices < 0] = 0
    final_behavior_values = behavior_values[behavior_indices]
    return final_behavior_values

def compute_correlation_map(pmd_obj,trace, exclusion=10, device='cpu', batch_size = 40):
    trace = torch.from_numpy(trace[exclusion:exclusion*-1]).to(device)
    trace -= torch.mean(trace)
    trace /= torch.linalg.norm(trace)

    height_iters = math.ceil(pmd_obj.shape[1] / batch_size)
    width_iters = math.ceil(pmd_obj.shape[2] / batch_size)

    corr_map = torch.zeros(pmd_obj.shape[1], pmd_obj.shape[2]).to(device)
    for k in tqdm(range(height_iters)):
        for j in range(width_iters):
            dim1 = slice(k * batch_size, min((k+1)*batch_size, pmd_obj.shape[1]))
            dim2 = slice(j * batch_size, min((j+1)*batch_size, pmd_obj.shape[2]))
            my_data = pmd_obj.getitem_tensor((slice(exclusion, pmd_obj.shape[0] - exclusion), dim1, dim2))
            my_data -= torch.mean(my_data, dim=0, keepdim=True)
            my_data /= torch.linalg.norm(my_data, dim=0, keepdim=True)
            my_data = torch.nan_to_num(my_data, nan = 0.0)

            corr_map[dim1, dim2] = torch.sum(my_data * trace[:, None, None], dim = 0)
    return corr_map.cpu().numpy()
            
            


In [36]:
## Pull our relevant correlations
behavior_values = extract_time_aligned_behavior(imaging_times,
                                                0,
                                                sl.wheel['times'].to_numpy(),
                                                sl.wheel['velocity'].to_numpy())


corr_map_amol = manual_mask * compute_correlation_map(hemocorr_pmd, behavior_values, exclusion=20000, device='cuda')
corr_map_joao = manual_mask * compute_correlation_map(joao_hemocorr, behavior_values, exclusion=20000, device='cuda')


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [01:20<00:00,  5.73s/it]


In [47]:
def plot_correlation_images(img_left, img_right, v_range):
    """
    img_left  : 2D numpy array
    img_right : 2D numpy array
    """

    if v_range is None:
        vmin, vmax = -0.1, 0.1  # fixed grayscale range
    else:
        vmin, vmax = v_range[0], v_range[1]

    fig, axes = plt.subplots(
        1, 2, figsize=(12, 5),
        gridspec_kw={"width_ratios": [1, 1], "wspace": 0.05}
    )

    # ---- Left image ----
    im = axes[0].imshow(img_left, cmap="gray", vmin=vmin, vmax=vmax)
    axes[0].set_title("corr(PMD Hemocorrected, wheel velocity)")
    axes[0].axis("off")

    # ---- Right image ----
    axes[1].imshow(img_right, cmap="gray", vmin=vmin, vmax=vmax)
    axes[1].set_title("corr(Wfield hemocorrected SVD, wheel velocity)")
    axes[1].axis("off")

    # Create a separate axis for colorbar
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label("Correlation")

    plt.tight_layout(rect=[0, 0, 0.9, 1])  # leave space for colorbar
    plt.savefig("Correlation_Over_Time_WheelVel.png", bbox_inches = "tight")
    plt.show()


In [50]:
iw = fpl.ImageWidget(data = [corr_map_amol, corr_map_joao])
iw.cmap = "gray"
iw.show()

In [49]:
plot_correlation_images(corr_map_amol, corr_map_joao, v_range = [-0.01, 0.08])